In [1]:
import os
import math
import numpy as np
import pandas as pd

In [2]:
# ===============================
# Parameters
# ===============================
sc = 'column30'  # solar collector potential
heat = 'column3'  # heating demand

min_sc_area = 10     # minimum installed solar collector area
sc_module_area = 2.5 # solar collector module area
min_sc_num = math.floor(min_sc_area / sc_module_area)



In [3]:
# ===============================
# Cost and gas functions
# ===============================
def cost(area):
    """Calculate CAPEX and OPEX without storage"""
    dic = {'cost_nos': 0, 'ins_cost_nos': 0, 'capex_nos': 0, 'opex_nos': 0}
    if area != 0:
        dic['cost_nos'] = 300 * area + 800
        dic['ins_cost_nos'] = 0.25 * dic['cost_nos']
        dic['capex_nos'] = dic['cost_nos'] + dic['ins_cost_nos']
        dic['opex_nos'] = 2.5 * area + 100
    return dic


def gas(h):
    """Calculate gas cost for heating demand h"""
    h = h / 0.8
    if h <= 2000:
        return 6.00 * 12 + (20.47 + 2.226) * h / 100
    elif h <= 10000:
        return 8.00 * 12 + (18.62 + 2.226) * h / 100
    elif h <= 30000:
        return 12.00 * 12 + (17.73 + 2.226) * h / 100
    elif h <= 100000:
        return 14.00 * 12 + (17.61 + 2.226) * h / 100
    elif h <= 300000:
        return 24.00 * 12 + (17.37 + 2.226) * h / 100
    else:
        return 60.00 * 12 + (17.2 + 2.226) * h / 100


# ===============================
# NPV functions
# ===============================
def initial_constants(dic, area):
    """Initialize constants for NPV without storage"""
    dic['gas_savings_nos'] = dic['gas_cost_nos'] - dic['remain_gas_cost_nos']
    result = cost(area)
    dic['capex_nos'] = result['capex_nos']
    dic['opex_nos'] = result['opex_nos']
    return dic


def npv_cal(dic, i):
    """Calculate discounted cash flow for year i"""
    gas_increase = 0.0449
    interest_rate = 0.0337
    c = dic['gas_savings_nos'] * (1 + gas_increase) ** i - dic['opex_nos']
    cur = c / (1 + interest_rate) ** i
    return cur


def npv(dic, years):
    """Calculate NPV without storage"""
    dic['n_nos_0'] = -1 * dic['capex_nos']
    dic['npv_nos'] = dic['n_nos_0']
    for i in range(1, years + 1):
        dic['n_nos_' + str(i)] = npv_cal(dic, i)
        dic['npv_nos'] += dic['n_nos_' + str(i)]
    return dic


# ===============================
# Solar-only simulation
# ===============================
def cal_sc(o_df, scale):
    """Simulate solar collector performance without storage"""
    df = o_df.copy()
    scaled_sc = 'scaled_sc_nos'
    o = 'over_supply_nos'
    us = 'under_supply_nos'
    need = 'need_nos'

    df[scaled_sc] = df[sc] * scale
    df[o] = np.where(df[scaled_sc] <= df[heat], 0, df[scaled_sc] - df[heat])
    df[us] = np.where(df[o] == 0, df[scaled_sc], df[heat])
    df[need] = df[heat] - df[us]

    return df, df[need].sum(), df[heat].sum() - df[need].sum()


# ===============================
# System installation optimization
# ===============================
def install_systems(df, max_area):
    """Optimize solar system configuration without storage"""
    max_num = int(max_area // sc_module_area)
    original_num = max_area / sc_module_area

    best_dic = {'heat': df[heat].sum()}
    best_dic['gas_cost_nos'] = gas(best_dic['heat'])
    best_dic['need_nos'] = best_dic['heat']
    best_dic['remain_gas_cost_nos'] = best_dic['gas_cost_nos']
    best_dic['num_sc_nos'] = 0
    best_dic['total_installed_area_nos'] = 0
    best_dic['saved_nos'] = 0
    best_dic = initial_constants(best_dic, 0)
    best_dic = npv(best_dic, 25)

    return_df = df.copy()

    if max_num < min_sc_num:
        return best_dic, df

    for i in range(max_num, min_sc_num - 1, -1):
        total_area = i * sc_module_area
        temp_df, need, saved = cal_sc(df, i / original_num)

        temp_dic = best_dic.copy()
        temp_dic['need_nos'] = need
        temp_dic['remain_gas_cost_nos'] = gas(temp_dic['need_nos'])
        temp_dic['num_sc_nos'] = i
        temp_dic['total_installed_area_nos'] = total_area
        temp_dic = initial_constants(temp_dic, total_area)
        temp_dic['saved_nos'] = saved
        temp_dic = npv(temp_dic, 25)

        if temp_dic['npv_nos'] >= best_dic['npv_nos']:
            best_dic = temp_dic.copy()
            return_df = temp_df.copy()

    return best_dic, return_df



In [11]:
def run_pipeline(debug=True):
    # File paths
    data_dir = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent"
    db_table_file = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\dbTable.csv"
    ts_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ts_kl.csv"
    building_use_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_kl_buildinguse.csv"
    building_area_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_kl.csv"

    # Load reference CSVs
    db_table = pd.read_csv(db_table_file)
    ts_map = pd.read_csv(ts_file)
    building_use_df = pd.read_csv(building_use_file)
    building_area_df = pd.read_csv(building_area_file)

    # Output summary
    summary = {
        'filename': [], 'building': [], 'buildinguse': [], 'roof_area': [],
        'num_sc_nos': [], 'total_installed_area_nos': [],
        'capex_nos': [], 'opex_nos': [],
        'need_nos': [], 'saved_nos': [],
        'npv_nos': []
    }
    for j in range(0, 26):
        summary[f'cashflow_year{j}_nos'] = []

    # Loop through all timeseries CSV files
    for file in os.listdir(data_dir):
        if not file.endswith(".csv"):
            continue

        file_uuid = file.replace(".csv", "")
        if debug:
            print(f"\n🔎 Processing file: {file} (uuid={file_uuid})")

        # Step 1: match file_uuid in dbTable.csv
        ts_row = db_table[db_table["tableName"].str.contains(file_uuid, na=False)]
        if ts_row.empty:
            print(f"  ❌ No match in dbTable.csv for {file_uuid}")
            continue
        dataIRI = ts_row.iloc[0]["dataIRI"]
        if debug:
            print(f"  ✅ Found dataIRI: {dataIRI}")

        # Step 2: match building in results_ts_kl.csv
        ts_row2 = ts_map[ts_map["Measurement"] == dataIRI]
        if ts_row2.empty:
            print(f"  ❌ No building found in results_ts_kl.csv for {dataIRI}")
            continue
        building = ts_row2.iloc[0]["building"]
        if debug:
            print(f"  ✅ Found building: {building}")

        # Step 3: get buildinguse
        bu_row = building_use_df[building_use_df["building"] == building]
        if bu_row.empty:
            print(f"  ⚠️ No buildinguse found, default to non_residential")
            buildinguse = "non_residential"
        else:
            buildinguse = bu_row.iloc[0]["buildinguse"]
        if debug:
            print(f"  ✅ Buildinguse: {buildinguse}")

        # Step 4: get roof area
        area_row = building_area_df[
            (building_area_df["building"] == building) &
            (building_area_df["Property"] == "Roof solar suitable area")
        ]
        if area_row.empty:
            print(f"  ❌ No roof area found in results_kl.csv for building {building}")
            continue
        roof_area = float(area_row.iloc[0]["Value"])
        if debug:
            print(f"  ✅ Roof area: {roof_area}")

        # Step 5: load timeseries CSV
        df = pd.read_csv(os.path.join(data_dir, file))
        if not {"column3", "column30"}.issubset(df.columns):
            print(f"  ❌ Missing required columns in {file}")
            continue
        if debug:
            print(f"  ✅ Timeseries file loaded with {len(df)} rows")

        # Step 6: run NPV calculation (without storage version)
        result, _ = install_systems(df, roof_area)
        if debug:
            print(f"  ✅ NPV calculated: {result['npv_nos']:.2f}")

        # Append results
        summary['filename'].append(file)
        summary['building'].append(building)
        summary['buildinguse'].append(buildinguse)
        summary['roof_area'].append(roof_area)
        summary['num_sc_nos'].append(result['num_sc_nos'])
        summary['total_installed_area_nos'].append(result['total_installed_area_nos'])
        summary['capex_nos'].append(result['capex_nos'])
        summary['opex_nos'].append(result['opex_nos'])
        summary['need_nos'].append(result['need_nos'])
        summary['saved_nos'].append(result['saved_nos'])
        summary['npv_nos'].append(result['npv_nos'])
        for j in range(0, 26):
            summary[f'cashflow_year{j}_nos'].append(result['n_nos_' + str(j)])

    # Save output
    out_file = "npv_results_without_storage.csv"
    pd.DataFrame(summary).to_csv(out_file, index=False)
    print(f"\n✅ Results saved to {out_file}")

In [12]:
if __name__ == "__main__":
    run_pipeline(debug=True)


🔎 Processing file: 0005fe0e-e72c-49cf-90e5-d8a75827695b.csv (uuid=0005fe0e-e72c-49cf-90e5-d8a75827695b)
  ✅ Found dataIRI: https://www.theworldavatar.com/kg/ontoubemmp/GridConsumption_b5c9b692-c724-4207-a0da-fecfaa872fe9/
  ✅ Found building: https://theworldavatar.io/kg/Building/4d11f8c0-f180-4201-a355-f07f25adcbf9
  ✅ Buildinguse: https://theworldavatar.io/kg/Domestic_b496bc5e-d44e-423d-a978-05d138dbcdea
  ✅ Roof area: 14.51
  ✅ Timeseries file loaded with 8760 rows
  ✅ NPV calculated: 0.00

🔎 Processing file: 000f3f26-9d48-4165-9075-92e1e0199331.csv (uuid=000f3f26-9d48-4165-9075-92e1e0199331)
  ✅ Found dataIRI: https://www.theworldavatar.com/kg/ontoubemmp/GridConsumption_5c923f30-47e8-47cd-aa58-4bf6ad534820/
  ✅ Found building: https://theworldavatar.io/kg/Building/16622925-c558-45e1-8653-106a4d8a183d
  ✅ Buildinguse: https://theworldavatar.io/kg/Domestic_048885bb-d1ba-4257-88cf-900bfa575887
  ✅ Roof area: 18.26
  ✅ Timeseries file loaded with 8760 rows
  ✅ NPV calculated: 0.00

🔎 P

In [13]:
def run_pipeline_PS(debug=True):
    # File paths
    data_dir = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\PS_TS\CEAAgent"
    db_table_file = r"D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\PS_TS\dbTable.csv"
    ts_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ts_ps.csv"
    building_use_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ps_buildinguse.csv"
    building_area_file = r"D:\c4e-jz713-a-tale-of-two-cities\Codes\results_ps.csv"

   # Load reference CSVs
    db_table = pd.read_csv(db_table_file)
    ts_map = pd.read_csv(ts_file)
    building_use_df = pd.read_csv(building_use_file)
    building_area_df = pd.read_csv(building_area_file)

    # Output summary
    summary = {
        'filename': [], 'building': [], 'buildinguse': [], 'roof_area': [],
        'num_sc_nos': [], 'total_installed_area_nos': [],
        'capex_nos': [], 'opex_nos': [],
        'need_nos': [], 'saved_nos': [],
        'npv_nos': []
    }
    for j in range(0, 26):
        summary[f'cashflow_year{j}_nos'] = []

    # Loop through all timeseries CSV files
    for file in os.listdir(data_dir):
        if not file.endswith(".csv"):
            continue

        file_uuid = file.replace(".csv", "")
        if debug:
            print(f"\n🔎 Processing file: {file} (uuid={file_uuid})")

        # Step 1: match file_uuid in dbTable.csv
        ts_row = db_table[db_table["tableName"].str.contains(file_uuid, na=False)]
        if ts_row.empty:
            print(f"  ❌ No match in dbTable.csv for {file_uuid}")
            continue
        dataIRI = ts_row.iloc[0]["dataIRI"]
        if debug:
            print(f"  ✅ Found dataIRI: {dataIRI}")

        # Step 2: match building in results_ts_kl.csv
        ts_row2 = ts_map[ts_map["Measurement"] == dataIRI]
        if ts_row2.empty:
            print(f"  ❌ No building found in results_ts_kl.csv for {dataIRI}")
            continue
        building = ts_row2.iloc[0]["building"]
        if debug:
            print(f"  ✅ Found building: {building}")

        # Step 3: get buildinguse
        bu_row = building_use_df[building_use_df["building"] == building]
        if bu_row.empty:
            print(f"  ⚠️ No buildinguse found, default to non_residential")
            buildinguse = "non_residential"
        else:
            buildinguse = bu_row.iloc[0]["buildinguse"]
        if debug:
            print(f"  ✅ Buildinguse: {buildinguse}")

        # Step 4: get roof area
        area_row = building_area_df[
            (building_area_df["building"] == building) &
            (building_area_df["Property"] == "Roof solar suitable area")
        ]
        if area_row.empty:
            print(f"  ❌ No roof area found in results_kl.csv for building {building}")
            continue
        roof_area = float(area_row.iloc[0]["Value"])
        if debug:
            print(f"  ✅ Roof area: {roof_area}")

        # Step 5: load timeseries CSV
        df = pd.read_csv(os.path.join(data_dir, file))
        if not {"column3", "column30"}.issubset(df.columns):
            print(f"  ❌ Missing required columns in {file}")
            continue
        if debug:
            print(f"  ✅ Timeseries file loaded with {len(df)} rows")

        # Step 6: run NPV calculation (without storage version)
        result, _ = install_systems(df, roof_area)
        if debug:
            print(f"  ✅ NPV calculated: {result['npv_nos']:.2f}")

        # Append results
        summary['filename'].append(file)
        summary['building'].append(building)
        summary['buildinguse'].append(buildinguse)
        summary['roof_area'].append(roof_area)
        summary['num_sc_nos'].append(result['num_sc_nos'])
        summary['total_installed_area_nos'].append(result['total_installed_area_nos'])
        summary['capex_nos'].append(result['capex_nos'])
        summary['opex_nos'].append(result['opex_nos'])
        summary['need_nos'].append(result['need_nos'])
        summary['saved_nos'].append(result['saved_nos'])
        summary['npv_nos'].append(result['npv_nos'])
        for j in range(0, 26):
            summary[f'cashflow_year{j}_nos'].append(result['n_nos_' + str(j)])

    # Save output
    out_file = "npv_results_without_storage_PS.csv"
    pd.DataFrame(summary).to_csv(out_file, index=False)
    print(f"\n✅ Results saved to {out_file}")

In [14]:
if __name__ == "__main__":
    run_pipeline_PS()


🔎 Processing file: 000be916-6d6b-4d4b-93b5-b71e0673dd0c.csv (uuid=000be916-6d6b-4d4b-93b5-b71e0673dd0c)
  ✅ Found dataIRI: https://www.theworldavatar.com/kg/ontoubemmp/GridConsumption_7c37ac21-88d3-48a7-8521-74acbf8d13e1/
  ✅ Found building: https://www.theworldavatar.com/kg/Building/f7fb0945-72a1-454e-81db-28c581e4bb45
  ✅ Buildinguse: https://www.theworldavatar.com/kg/Domestic_eba852df-8f7e-477b-8667-742046a333e5
  ✅ Roof area: 0.0
  ✅ Timeseries file loaded with 8760 rows
  ✅ NPV calculated: 0.00

🔎 Processing file: 0010a97b-935a-445e-ba0b-68267493f27d.csv (uuid=0010a97b-935a-445e-ba0b-68267493f27d)
  ✅ Found dataIRI: https://www.theworldavatar.com/kg/ontoubemmp/GridConsumption_07f02366-311f-48a6-a91d-92615c712b22/
  ✅ Found building: https://www.theworldavatar.com/kg/Building/b549eb29-7c38-40b9-a19d-46b067bb2a86
  ✅ Buildinguse: https://www.theworldavatar.com/kg/Domestic_62b3604b-47a5-4beb-9ed0-c43c3866622d
  ✅ Roof area: 86.38
  ✅ Timeseries file loaded with 8760 rows
  ✅ NPV cal